# 🚨 Alert Pipeline — Manufacturing Anomaly Alerts

Reads anomaly results from `gold_sensor_anomalies` and `gold_defect_anomalies`,
creates unified alert tables for Power BI dashboards.

**Writes to:** `gold_alerts`, `gold_alert_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, count, avg,
    max as spark_max, min as spark_min, concat_ws,
    to_date, date_format, row_number
)
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load anomaly Gold tables
sensor_anomalies = spark.read.table('gold_sensor_anomalies')
defect_anomalies = spark.read.table('gold_defect_anomalies')

print(f'Sensor anomaly rows: {sensor_anomalies.count()}')
print(f'Defect anomaly rows: {defect_anomalies.count()}')

## 1. Create unified alert table

Combine sensor anomalies and defect anomalies into a single `gold_alerts` table.

In [ ]:
# Sensor alerts — only actual anomalies
sensor_alerts = (
    sensor_anomalies
    .filter(col('is_anomaly') == 1)
    .select(
        col('reading_id').alias('source_id'),
        lit('sensor').alias('alert_category'),
        col('anomaly_channel').alias('alert_type'),
        col('machine_id'),
        lit(None).cast('string').alias('production_line'),
        col('reading_timestamp').alias('alert_timestamp'),
        col('reading_date').alias('alert_date'),
        col('shift'),
        col('severity'),
        col('anomaly_score').alias('confidence_score'),
        concat_ws(' | ',
            concat_ws('=', lit('temp'), col('temperature').cast('string')),
            concat_ws('=', lit('press'), col('pressure').cast('string')),
            concat_ws('=', lit('vib'), col('vibration').cast('string'))
        ).alias('detail'),
        col('detection_timestamp')
    )
)

# Defect alerts — only flagged anomalies
defect_alerts = (
    defect_anomalies
    .filter(col('anomaly_type') != 'normal')
    .select(
        col('batch_id').alias('source_id'),
        lit('production').alias('alert_category'),
        col('anomaly_type').alias('alert_type'),
        lit(None).cast('string').alias('machine_id'),
        col('production_line'),
        col('detection_timestamp').alias('alert_timestamp'),
        col('production_date').alias('alert_date'),
        col('shift'),
        col('severity'),
        col('defect_zscore').alias('confidence_score'),
        concat_ws(' | ',
            concat_ws('=', lit('defect_rate'), col('defect_rate').cast('string')),
            concat_ws('=', lit('yield'), col('yield_pct').cast('string')),
            concat_ws('=', lit('downtime_min'), col('downtime_minutes').cast('string'))
        ).alias('detail'),
        col('detection_timestamp')
    )
)

# Union all alerts
gold_alerts = sensor_alerts.unionByName(defect_alerts)

gold_alerts.write.mode('overwrite').format('delta').saveAsTable('gold_alerts')
print(f'Gold alerts written: {gold_alerts.count()} rows')
print(f'  - Sensor alerts: {sensor_alerts.count()}')
print(f'  - Defect alerts: {defect_alerts.count()}')

## 2. Create alert summary table

Daily/shift-level summary for Power BI dashboard KPI cards.

In [ ]:
# Daily summary by category and severity
gold_alert_summary = (
    gold_alerts
    .groupBy('alert_date', 'alert_category', 'severity', 'shift')
    .agg(
        count('*').alias('alert_count'),
        avg('confidence_score').alias('avg_confidence'),
    )
    .withColumn('summary_timestamp', current_timestamp())
)

gold_alert_summary.write.mode('overwrite').format('delta').saveAsTable('gold_alert_summary')
print(f'Gold alert summary written: {gold_alert_summary.count()} rows')

In [ ]:
# Optimize
spark.sql('OPTIMIZE gold_alerts')
spark.sql('OPTIMIZE gold_alert_summary')
print('Alert tables optimized')

In [ ]:
# Final summary
print('\n=== Alert Pipeline Summary ===')
print(f'Total alerts: {gold_alerts.count()}')

severity_counts = gold_alerts.groupBy('severity').count().collect()
for row in severity_counts:
    print(f'  {row["severity"]}: {row["count"]}')

print('\nTables created:')
print('  - gold_alerts (unified alert log)')
print('  - gold_alert_summary (daily KPI summary)')